In [13]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

In [14]:
words = open(file='./data/names.txt',mode='r').read().splitlines()
print(len(words))
print(max(len(w) for w in words))
print(words[:8])

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


In [15]:
chars = sorted(list(set(''.join(words))))
stoi = {k: v+1 for v, k in enumerate(chars)}
stoi['.'] = 0
itos = {k: v for v, k in stoi.items()}
print(len(stoi))
vocab_size = len(stoi)

27


In [ ]:
### builde the dataset
block_size = 3
X, Y = [], []
def build_dataset(data):
    for w in data:
        context = [0] *3 
        chs = w + '.'
        for ch in chs:
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)
np.random.seed(42)
words = np.random.permutation(words)

n1 = int(len(words) * 0.8)
n2 = int(len(words) * 0.9)

Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])

In [17]:
def cmp(s,dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.isclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s: 15s} | exact: {str(ex): 5s} | approximate: {str(app): 5s} | maxdiff: {str(maxdiff)}')

In [45]:
nemd = 10
n_hidden = 100
g = torch.Generator().manual_seed(0)
C = torch.randn(size = (vocab_size, nemd),generator= g)

W1 = torch.randn(size = (block_size * nemd, n_hidden), generator= g) * (5/3) / ((block_size * nemd) ** 0.5)
b1 = torch.randn(size = (n_hidden,) ,generator= g) * 0.1
 
W2 = torch.randn(size = (n_hidden, vocab_size), generator= g) * 0.1
b2 = torch.randn(size = (vocab_size,), generator= g) * 0.1

bngain = torch.ones(size=(1,n_hidden))*0.1 + 0.1
bnbias = torch.zeros(size=(1,n_hidden))

parameters = [C, W1, b1, W2, b2, bngain, bnbias]
for p in parameters:
    p.requires_grad = True

In [46]:
batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (n,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]
print(Xtr.shape, Ytr.shape) # (32033, 3) (32033,)
print(Xb.shape, Yb.shape) # (32, 3) (32,)

torch.Size([182671, 3]) torch.Size([182671])
torch.Size([32, 3]) torch.Size([32])


In [48]:
emb = C[Xb]
embcat = emb.view(emb.shape[0], -1)
hprebn = embcat @ W1 + b1 ### shape: (32, 300) @ (300, 100) + (100,) -> (32, 100)
# batchnorm
bnmeani = 1/n * hprebn.sum(0, keepdim=True)### shape: (1, 100)
bnvar = 1/n * ((hprebn - bnmeani)**2).sum(0, keepdim=True)### shape: (1, 100)
bndiff = hprebn - bnmeani
bndiff2 = bndiff**2
bnvar = 1/(n - 1) * bndiff2.sum(0, keepdim=True) ### shape: (1, 100)
bnvar_inv = (bnvar + 1e-5)**-0.5
bnraw = bndiff * bnvar_inv
hpreact = bngain * bnraw + bnbias
h = torch.tanh(hpreact)
logits = h @ W2 + b2### shape: (32, 100) @ (100, 27) + (27,) -> (32, 27)
logits_max = torch.max(logits, dim=1, keepdim=True)[1]
norm_logits = logits - logits_max
counts = norm_logits.exp() ### shape: (32, 27)
counts_sum = counts.sum(1, keepdim=True) ### shape: (32, 1)
counts_sum_inv = counts_sum**-1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(batch_size), Yb].mean()

### Pytorch backward pass
for p in parameters:
    p.grad = None
# for t in [logprobs,probs, counts, counts_sum, counts_sum_inv, norm_logits, logits_max,logits, h, hpreact, bnraw, bnvar_inv, bnvar, bndiff2, bndiff, bnmeani, hprebn, embcat, emb]:
#     t.retain_grad()### 
tensors = [logprobs, probs, counts, counts_sum, counts_sum_inv, norm_logits,
           logits_max, logits, h, hpreact, bnraw, bnvar_inv, bnvar,
           bndiff2, bndiff, bnmeani, hprebn, embcat, emb]

for t in tensors:
    if t.requires_grad:
        t.retain_grad()

loss.backward()
loss

tensor(3.2573, grad_fn=<NegBackward0>)

In [34]:
a = torch.randn(size=(3,3))
torch.max(a, dim=1, keepdim=True)

torch.return_types.max(
values=tensor([[0.6328],
        [0.7625],
        [1.8965]]),
indices=tensor([[2],
        [2],
        [1]]))

In [ ]:
a = torch.tensor(4.0, requires_grad=True)
b = a * 2
b.retain_grad()
c = b * b * 3
c.backward()
print(b.grad)

tensor(48.)
